<div align="center">

# NNDL Final Project: Flight Delay Forecasting

**Kevin Brugnera · Sara Pasquato · Libero Pollini**

IDs: 2196578 · (inserite) · 2206131

</div>

---

## LSTM model

First and foremost, we explore the 2022 chain dataset.

### Imports

In [1]:
#import pandas as pd

import kagglehub # to download original dataset

# time performance
import time

# torch libraries
import torch
import torch.nn as nn
from torch.utils.data import Subset
import torch.optim as optim
from torch.utils.data import DataLoader

# for saving 'checkpoints' / results
import pickle
import copy

# metrics for testing
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# check if GPU available
gpu_av=torch.cuda.is_available()

# for reproducibility
SEED = 42
torch.manual_seed(SEED)
print("GPU available:", gpu_av)
if gpu_av:
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

GPU available: True


In [2]:
# from google.colab import files # opens interactive window to choose file(s) to upload files
# import os

# uploaded = files.upload()

# for name in uploaded.keys():
#     print

In [3]:
# define column names of original tabular dataset as global variables
# (for dataset download later)
DATE_COLS=["FL_DATE"]
DATETIME_COLS=["CRS_DEP_TIME", "CRS_ARR_TIME", "DEP_TIME",
               "ARR_TIME", "WHEELS_OFF", "WHEELS_ON", ]
TIMEDELTA_MINS_COLS=["DEP_DELAY", "ARR_DELAY", "TAXI_OUT", "TAXI_IN", "CRS_ELAPSED_TIME",
                     "ACTUAL_ELAPSED_TIME", "AIR_TIME",	]
INT_COLS=["OP_CARRIER_FL_NUM", "FLIGHTS", "MONTH", "DAY_OF_MONTH",
          "DAY_OF_WEEK", "ORIGIN_INDEX", "DEST_INDEX"]
STR_COLS=["OP_CARRIER", "ORIGIN", "DEST"]
FLOAT_COLS=["O_TEMP", "O_PRCP", "O_WSPD", "D_TEMP", "D_PRCP", "D_WSPD", "O_LATITUDE",
             "O_LONGITUDE", "D_LATITUDE", "D_LONGITUDE"]
size_set_check=set(DATE_COLS+DATETIME_COLS+TIMEDELTA_MINS_COLS+INT_COLS+STR_COLS+FLOAT_COLS)
print(f"Total individual features (should be 34): {len(size_set_check)}")

Total individual features (should be 34): 34


In [4]:
# to reload in case data_loader.py has been changed in the meantime:
#import importlib
#import data_loader
#importlib.reload(data_loader)

# only use in local:
#from data_loader import download_dataset, load_dataset_pytorch

In [5]:
from sklearn.preprocessing import (
    LabelEncoder,
)  # to encode categorical features as integers

from torch.utils.data import TensorDataset # for saving after removing features

import os  # to save dataset later

import shutil # to move dataset to correct destination folder


def download_dataset(
    year_start,
    year_end_exd, # end year (excluded)
    origin_path="flnny123/mfddmulti-modal-flight-delay-dataset/versions/4",
    mode="tabular",  # or "sequential" for pre-made chains
    output_dir_seq="data/chain/",
):

    dest_paths = []
    for year in range(year_start, year_end_exd):
        print(f"Downloading year {year} data...")
        if mode == "tabular": # tabular dataset download
            origin_path_year = (
                "Aeolus/Flight_Tab/flight_with_weather_" + str(year) + ".csv"
            )
            dest_path_year = kagglehub.dataset_download(
                origin_path, path=origin_path_year
            )
            dest_paths.append(dest_path_year)

        # dest_path_year = dest_path+'flight_with_weather_'+str(year)+'.csv' # destination path
        elif mode == "sequential": # sequential (chains) dataset download
            for split in ["train", "val", "test"]:
                origin_path_year_split = f"Aeolus/Flight_chain/chain_data_{year}/{split}_flight_chain_{year}.pt"
                final_path = os.path.join(
                    output_dir_seq + str(year), f"{split}_flight_chain_{year}.pt"
                )
                if os.path.exists(final_path):
                    print(f"Path {final_path} already exists! Skipping it")
                    continue
                dest_path_year = kagglehub.dataset_download(
                    origin_path, path=origin_path_year_split
                )
                os.makedirs(output_dir_seq + str(year), exist_ok=True)
                shutil.move(dest_path_year, final_path)
                dest_path_year = final_path
                dest_paths.append(final_path)

                print(f"(File(s) available at {dest_path_year}).")

    return dest_paths


def load_dataset_pytorch(
    year, file_path="data/chain/"
):  # for sequential when directly available
    split_types = ["train", "val", "test"]

    loaded_data = {}

    for split in split_types:
        full_file_path = file_path + f"{year}/{split}_flight_chain_{year}.pt"
        # loaded_data[split] = torch.load(full_file_path, weights_only=False)
        dataset = torch.load(full_file_path, weights_only=False)

        # Slice dense tensor to remove the last, constant feature (FLIGHTS), which is always equal to 1
        dense = dataset.tensors[0]  # [N, seq_len, 7]
        dense = dense[
            :, :, :-1
        ].clone()  # now [N, seq_len, 6]. Clone to make sure not even the last column is copied into RAM
        # Rebuild dataset with the same other tensors
        loaded_data[split] = TensorDataset(dense, *dataset.tensors[1:])

        print(f"--- Read file: (split: {split}, year: {year}) ---")

    return loaded_data

### Load dataset

In [6]:
year = 2022


In [7]:
file_path = download_dataset(
    year, year + 1, mode="sequential"
)  # or "sequential" for pre-made chains)


100%|██████████| 877M/877M [00:17<00:00, 51.9MB/s]


(File(s) available at data/chain/2022/train_flight_chain_2022.pt).


100%|██████████| 331M/331M [00:06<00:00, 50.3MB/s]


(File(s) available at data/chain/2022/val_flight_chain_2022.pt).


100%|██████████| 318M/318M [00:06<00:00, 49.9MB/s]

(File(s) available at data/chain/2022/test_flight_chain_2022.pt).


In [8]:
# load chain datasets with pytorch
# split_types = ["train", "val", "test"]

# loaded_data = {}

# for split in split_types:
#     file_path = f"data/chain/{year}/{split}_flight_chain_{year}.pt"
#     loaded_data[split] = torch.load(file_path, weights_only=False)

#     print(f"--- File: (split: {split}, year: {year}) ---")
#     print("Data Type:", type(loaded_data[split]))

loaded_data=load_dataset_pytorch(year=2022)

--- Read file: (split: train, year: 2022) ---
--- Read file: (split: val, year: 2022) ---
--- Read file: (split: test, year: 2022) ---


#### Remove rows with outlier targets in training dataset

In [9]:
def remove_outliers_percentile_chains(dataset, lower=0.01, upper=0.99):
    """
    Removes whole CHAIN where any valid (non-padded) individual delay step (flight)
    falls outside the [lower, upper] percentile range.
    """
    delays = dataset.tensors[4]       # [N, seq_len, 2]
    valid_lens = dataset.tensors[3]   # [N]
    seq_len = delays.shape[1]

    mask = torch.arange(seq_len).unsqueeze(0) < valid_lens.unsqueeze(1)  # [N, seq_len]
    valid_delays = delays[mask].float()  # only valid entries, per column

    lower_bound = torch.quantile(valid_delays, lower, dim=0)  # [2]
    upper_bound = torch.quantile(valid_delays, upper, dim=0)  # [2]

    # per-sample: True if ALL valid steps are within bounds for BOTH target columns
    within_bounds = ((delays >= lower_bound) & (delays <= upper_bound)).all(dim=-1)  # [N, seq_len]
    within_bounds = within_bounds | ~mask  # ignore padded positions (always count as "within")
    keep_sample = within_bounds.all(dim=1)  # [N] -- True if no valid step is an outlier

    n_before = len(dataset)
    filtered_tensors = tuple(t[keep_sample] for t in dataset.tensors)
    n_after = filtered_tensors[0].shape[0]

    percentage_dropped = 100 * (n_before - n_after) / n_before
    print(f"Dropped {percentage_dropped:.2f} % of chains because outliers.")

    return TensorDataset(*filtered_tensors)

In [10]:
loaded_data["train"] = remove_outliers_percentile_chains(loaded_data["train"])

Dropped 3.32 % of chains because outliers.


#### Exploration and feature scaling

In [11]:
# On targets
train_dataset = loaded_data["train"]
all_delays = train_dataset.tensors[4]      # [N, seq_len, 2]
all_valid_lens = train_dataset.tensors[3]  # [N]

seq_len = all_delays.shape[1]
mask = torch.arange(seq_len).unsqueeze(0) < all_valid_lens.unsqueeze(1)  # [N, seq_len]

valid_delays = all_delays[mask].float()  # [total_valid_steps, 2] -- only as big as valid entries, no dataset copy

delay_mean = valid_delays.mean(dim=0)  # shape [2]
delay_std = valid_delays.std(dim=0)    # shape [2]

print(f"Delay mean (ARR, DEP): {delay_mean}")
print(f"Delay std (ARR, DEP): {delay_std}")

# --- 2. Scale/unscale helpers (operate on tensors on-the-fly, no dataset copy) ---
def scale_tensors(targets, mean=delay_mean, std=delay_std):
    return (targets - mean.to(targets.device)) / std.to(targets.device)

def unscale_tensors(scaled, mean=delay_mean, std=delay_std):
    return scaled * std.to(scaled.device) + mean.to(scaled.device)

Delay mean (ARR, DEP): tensor([3.6285, 8.9685])
Delay std (ARR, DEP): tensor([31.2527, 28.3377])


In [48]:
delay_threshold_scaled = scale_tensors(torch.tensor([15.0, 15.0]))  # 15-min threshold in SCALED units
print(delay_threshold_scaled)

tensor([0.3639, 0.2128])


In [12]:
# On dense features
all_dense = train_dataset.tensors[0]  # [N, seq_len, 6]
valid_dense = all_dense[mask].float()  # [total_valid_steps, 6], reuses same mask as before

dense_mean = valid_dense.mean(dim=0)  # [6]
dense_std = valid_dense.std(dim=0)    # [6]

In [13]:
print(dense_mean)
print(dense_std)

tensor([16.1060, 16.7422,  0.0957,  0.0987, 12.5930, 13.1573])
tensor([10.7501, 10.8325,  0.7448,  0.7677,  8.6650,  8.7176])


Samples are fewer than those reported in [Aeolus](https://arxiv.org/pdf/2510.26616) tabular dataset because they filtered to retain only coherent flight chains operated by the same aircraft (see Table 8, page 17).

In [14]:
N_train = len(loaded_data["train"])
N_val = len(loaded_data["val"])
N_test = len(loaded_data["test"])

N_chains = N_train + N_val + N_test

print(f"Total number of flight chains: {N_chains}")
print(f"Train samples: {N_train} ({N_train*100/N_chains:.2f}%)")
print(f"Validation samples: {N_val} ({N_val*100/N_chains:.2f}%)")
print(f"Test samples: {N_test} ({N_test*100/N_chains:.2f}%)")


Total number of flight chains: 5093718
Train samples: 2886190 (56.66%)
Validation samples: 1126506 (22.12%)
Test samples: 1081022 (21.22%)


Example of a flight chain sample

In [15]:
# 1. Extract the first sample from the training dataset
sample = loaded_data["train"][0]

print(f"Sample Type: {type(sample)}")
print(f"Total components in the sample tuple: {len(sample)}\n")
print("-" * 60)

# 2. Unpack each component of the 5-element tuple into properly named variables
dense_feat, sparse_feat, labels, valid_lens, delays = sample

# 3. Print details and relative values for each component with descriptive labels based on source code
print("1. Dense Features (Continuous / Meteorological Features):")
print(f"   - Shape: {dense_feat.shape} (Sequence Length x 6)")
print(
    f"   - Cols: ['O_TEMP', 'D_TEMP', 'O_PRCP', 'D_PRCP', 'O_WSPD', 'D_WSPD']"
)
print(f"    - Values:\n{dense_feat}\n")

print("2. Sparse Features (Categorical / Temporal Features):")
print(f"   - Shape: {sparse_feat.shape} (Sequence Length x 8)")
print(
    f"   - Cols: ['MONTH', 'DAY_OF_WEEK', 'CRS_ARR_TIME_HOUR', 'CRS_DEP_TIME_HOUR', 'ORIGIN_INDEX', 'DEST_INDEX', 'OP_CARRIER', 'OP_CARRIER_FL_NUM']"
)
print(f"    - Values:\n{sparse_feat}\n")

print("3. Binary Labels (Flight Delay Indicators > 15 mins):")
print(f"   - Shape: {labels.shape} (Sequence Length x 2)")
print(f"   - [(ARR_DELAY > 15), (DEP_DELAY > 15)]")
print(f"   - Values:\n{labels}\n")

print("4. Valid Sequence Lengths (Metadata):")
print(
    f"   - Description: Effective number of valid flights in the chain before padding"
)
print(f"   - Shape: {valid_lens.shape}")
print(f"   - Values: {valid_lens}\n")

print("5. Raw Delays (Ground Truth):")
print(f"   - Shape: {delays.shape} (Sequence Length x 2)")
print(f"   - Cols: ['ARR_DELAY', 'DEP_DELAY']")
print(f"   - Values:\n{delays}")
print("-" * 60)

Sample Type: <class 'tuple'>
Total components in the sample tuple: 5

------------------------------------------------------------
1. Dense Features (Continuous / Meteorological Features):
   - Shape: torch.Size([6, 6]) (Sequence Length x 6)
   - Cols: ['O_TEMP', 'D_TEMP', 'O_PRCP', 'D_PRCP', 'O_WSPD', 'D_WSPD', 'FLIGHTS']
    - Values:
tensor([[ 6.7000, 19.4000,  0.0000,  0.0000,  9.4000, 14.8000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000]])

2. Sparse Features (Categorical / Temporal Features):
   - Shape: torch.Size([6, 8]) (Sequence Length x 8)
   - Cols: ['MONTH', 'DAY_OF_WEEK', 'CRS_ARR_TIME_HOUR', 'CRS_DEP_TIME_HOUR', 'ORIGIN_INDEX', 'DEST_INDEX', 'OP_CARRIER', 'OP_CARRIER_FL_NUM']
    - Values:
tensor([[  

In [16]:
print(valid_lens)

tensor(1)


### LSTM model

#### Metrics

In [40]:
#see paper for definitions of metrics
import numpy as np

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve
)


def find_optimal_threshold(y_true, y_score): # for more realistic (and optimistic) calculation of F1 / recall,
# instead of fixed delay defined at 15 mins
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    J = tpr - fpr
    optimal_idx = np.argmax(J)
    return thresholds[optimal_idx]

def compute_all_metrics(pred_delays, true_delays, true_labels, valid_lens, dep_threshold=None, arr_threshold=None):
    """
    Compute regression (MSE, MAE) and classification (Accuracy, Precision, Recall, F1, AUC)
    for arrival and departure delays, masking out padded time steps.

    Args:
        pred_delays: Tensor [batch, seq_len, 2] - predicted delays (arr, dep)
        true_delays: Tensor [batch, seq_len, 2] - ground truth delays (arr, dep)
        true_labels: Tensor [batch, seq_len, 2] - binary labels (1 if delay > 15 min)
        valid_lens:  Tensor [batch] - number of valid (non-padded) steps per sequence

    Returns:
        dict: nested dict with metrics for arrival ('arr') and departure ('dep'),
              each containing regression and classification metrics.
    """
    batch_size, seq_len = pred_delays.shape[0], pred_delays.shape[1]
    device = pred_delays.device

    # Mask: True for valid positions, False for padding
    mask = torch.arange(seq_len, device=device).expand(
        batch_size, seq_len
    ) < valid_lens.unsqueeze(1)  # [batch, seq_len]

    # Flatten only valid entries for each target column
    pred_arr = pred_delays[:, :, 0][mask].flatten().cpu().numpy()
    pred_dep = pred_delays[:, :, 1][mask].flatten().cpu().numpy()
    true_arr = true_delays[:, :, 0][mask].flatten().cpu().numpy()
    true_dep = true_delays[:, :, 1][mask].flatten().cpu().numpy()
    label_arr = true_labels[:, :, 0][mask].flatten().cpu().numpy()
    label_dep = true_labels[:, :, 1][mask].flatten().cpu().numpy()

    # Helper for classification metrics
    def clf_metrics(y_true, y_pred, y_score):
        """
        Compute accuracy, precision, recall, F1 from binary predictions,
        and AUC from raw scores.
        """
        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        try:
            auc = roc_auc_score(y_true, y_score)
            # y_score are raw scores (obtained through regression-task model) for AUC
        except ValueError:
            auc = 0.5  # fallback if only one class present
        return acc, prec, rec, f1, auc

    # Regression metrics
    mse_arr = mean_squared_error(true_arr, pred_arr)
    mae_arr = mean_absolute_error(true_arr, pred_arr)
    mse_dep = mean_squared_error(true_dep, pred_dep)
    mae_dep = mean_absolute_error(true_dep, pred_dep)

    # Classification: "delay" threshold at variable time
    if arr_threshold is None:
        arr_threshold = find_optimal_threshold(label_arr, pred_arr)
        print("Using best threshold for arrival metrics...")
    if dep_threshold is None:
        dep_threshold = find_optimal_threshold(label_dep, pred_dep)
        print("Using best threshold for departure metrics...")

    pred_bin_arr = (pred_arr > arr_threshold).astype(int)
    pred_bin_dep = (pred_dep > dep_threshold).astype(int)

    acc_arr, prec_arr, rec_arr, f1_arr, auc_arr = clf_metrics(
        label_arr, pred_bin_arr, pred_arr
    )
    acc_dep, prec_dep, rec_dep, f1_dep, auc_dep = clf_metrics(
        label_dep, pred_bin_dep, pred_dep
    )

    metrics = {
        "arr_reg": {"MSE": mse_arr, "MAE": mae_arr},
        "dep_reg": {"MSE": mse_dep, "MAE": mae_dep},
        "arr_clf": {
            "Accuracy": acc_arr,
            "Precision": prec_arr,
            "Recall": rec_arr,
            "F1": f1_arr,
            "AUC": auc_arr,
        },
        "dep_clf": {
            "Accuracy": acc_dep,
            "Precision": prec_dep,
            "Recall": rec_dep,
            "F1": f1_dep,
            "AUC": auc_dep,
        },
    }
    return metrics

#### Model definition

##### First LSTM model on fixed length chains (6 flights)

In [18]:
# # Filtering

# # 1. Define the exact target length required
# target_length = 5

# # 2. Initialize a dictionary to store the filtered datasets
# filtered_loaded_data = {}

# # 3. Iterate through each split ('train', 'val', 'test') in the loaded dataset
# for split_name, dataset in loaded_data.items():
#     print(f"Processing split: {split_name} (Original samples: {len(dataset)})")

#     # Filter samples where the sequence length (dense_feat shape[0]) equals target_length
#     filtered_samples = [
#         sample for sample in dataset
#         if sample[0].shape[0] == target_length
#     ]

#     print(f"-> Filtered samples (length == {target_length}): {len(filtered_samples)}")



In [19]:
#     # If samples match the condition, reconstruct into a TensorDataset
#     if len(filtered_samples) > 0:
#         dense_list = torch.stack([s[0] for s in filtered_samples])
#         sparse_list = torch.stack([s[1] for s in filtered_samples])
#         labels_list = torch.stack([s[2] for s in filtered_samples])
#         valid_lens_list = torch.stack([s[3] for s in filtered_samples])
#         delays_list = torch.stack([s[4] for s in filtered_samples])

#         filtered_loaded_data[split_name] = TensorDataset(
#             dense_list, sparse_list, labels_list, valid_lens_list, delays_list
#         )
#     else:
#         # Keep an empty or None placeholder if no samples match
#         filtered_loaded_data[split_name] = None

# print("\nFiltering complete. All splits are stored in 'filtered_loaded_data'.")

##### LSTM with NN embedding of sparse features + dropout + linear regression layer

In [20]:
all_sparse = train_dataset.tensors[1]  # [N, seq_len, 8]
sparse_cardinalities = [int(all_sparse[:, :, i].max().item()) + 1 for i in range(all_sparse.shape[-1])]
# maximum number of distinct values for each categorical (sparse) feature

In [21]:
print(sparse_cardinalities)

[12, 7, 24, 24, 322, 322, 17, 6768]


In [22]:
class FlightChainLSTM(nn.Module):
    def __init__(
        self, dense_input_dim, sparse_cardinalities, embed_dim, hidden_dim, dropout_prob, output_dim=2
    ):

        super().__init__()

        # one embedding table per sparse column
        # constant embedding dimension for simplicity (and to avoid overfitting)
        self.embeddings = nn.ModuleList(
            [
                nn.Embedding(num_embeddings=card, embedding_dim=embed_dim)
                for card in sparse_cardinalities
            ]
        )

        total_sparse_embed_dim = embed_dim * len(sparse_cardinalities)
        self.total_input_dim = dense_input_dim + total_sparse_embed_dim

        self.hidden_dim = hidden_dim
        self.dropout_prob= dropout_prob

        # LSTM Layer configured with hidden units and batch_first=True
        # Input shape expected: [batch_size, sequence_length (6), total_input_dim]
        self.lstm = nn.LSTM(
            input_size=self.total_input_dim,  # chain lentgh, always 6
            hidden_size=self.hidden_dim,  # number of features in the hidden state h            num_layers=1,  # number of "stacked" LSTM layers (kept simple)
            batch_first=True, # LSTM input / output structure (which dimension first)
        )

        self.dropout = nn.Dropout(self.dropout_prob) # dropout layer for regularisation

        # Fully Connected Linear Regression Layer to map LSTM outputs to the target delay values
        self.regressor = nn.Linear(self.hidden_dim, output_dim)

    def forward(self, dense_feat, sparse_feat):

        # sparse_feat: [batch, seq_len, num_sparse_cols], integer dtype
        embedded = [
            emb(sparse_feat[:, :, i].long())  # [batch, seq_len, embed_dim]
            for i, emb in enumerate(self.embeddings)
        ]
        sparse_embed = torch.cat(
            embedded, dim=-1
        )  # [batch, seq_len, embed_dim * num_cols]
        x = torch.cat((dense_feat, sparse_embed), dim=-1) # concatenate dense and sparse (embedded) features
        lstm_out, (hn, cn) = self.lstm(x)
        lstm_out = self.dropout(lstm_out)
        predictions = self.regressor(lstm_out)

        return predictions

#### Hyperparameters & Configuration

In [56]:
DENSE_DIM = 6  # O_TEMP, D_TEMP, O_PRCP, D_PRCP, O_WSPD, D_WSPD.
# however no "FLIGHTS"
SPARSE_DIM = 8  # "MONTH","DAY_OF_WEEK","CRS_ARR_TIME_HOUR","CRS_DEP_TIME_HOUR",
#"ORIGIN_INDEX", "DEST_INDEX","OP_CARRIER", "OP_CARRIER_FL_NUM",
HIDDEN_UNITS = 32  # original 32
EMBED_DIM=4 # feature space embedding (NN) dimensionality
OUTPUT_REGRESSION_DIM = 2  # ARR_DELAY and DEP_DELAY

BATCH_SIZE = 64  # original 64
LEARNING_RATE = 0.001  # original 0.001
NUM_EPOCHS = 50  # original 50
PATIENCE = 5 # max number of iteration with no improvement in validation loss, for early stopping
WEIGTH_DECAY=1e-4 # used in AdamW optimizer
DROPOUT_PROB = 0.3 # in dropout layer
HUBER_LOSS_THRESH=1.0 # Huber loss threshold where MSE->MAE
POS_WEIGTH = 4.5 # class imbalance: delayed flights weigthed more by this factor

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS=0

print(f"Using device: {DEVICE}")

# 2. Create DataLoaders from your filtered splits - NO: use original (already padded/truncated) dataset
# train_loader = DataLoader(filtered_loaded_data['train'], batch_size=BATCH_SIZE, shuffle=True)
# val_loader = DataLoader(filtered_loaded_data['val'], batch_size=BATCH_SIZE, shuffle=False)

# load a fraction of dataset (can be 1.0)
fraction = 1.0 # 0.1  # for trial runs ---> about 500_000 samples

train_indices = torch.randperm(len(loaded_data["train"]))[
    : int(fraction * len(loaded_data["train"]))
]
val_indices = torch.randperm(len(loaded_data["val"]))[
    : int(fraction * len(loaded_data["val"]))
]
test_indices = torch.randperm(len(loaded_data["test"]))[
    : int(fraction * len(loaded_data["test"]))
]

train_loader = DataLoader(
    Subset(loaded_data["train"], train_indices),
    batch_size=BATCH_SIZE,
    shuffle=True,  # cant hurt in training
    num_workers=NUM_WORKERS,
    pin_memory=True,  # for faster but more memory consuming training
)
val_loader = DataLoader(
    Subset(loaded_data["val"], val_indices),
    batch_size=BATCH_SIZE,
    shuffle=False,  # bc True unnecessary, would add computation overhead and lose reproducibility
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
test_loader = DataLoader(
    Subset(loaded_data["test"], test_indices),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

Using device: cuda


In [57]:
# 3. Instantiate the Model, Loss, and Optimizer
model = FlightChainLSTM(
    dense_input_dim=DENSE_DIM,
    sparse_cardinalities=sparse_cardinalities,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_UNITS,
    dropout_prob=DROPOUT_PROB,
    output_dim=OUTPUT_REGRESSION_DIM,
).to(DEVICE)

# MSE Loss is standard for regression tasks (predicting continuous delay values)
# criterion = nn.MSELoss()


In [58]:
# sanity check on number of model parameters
print(sum(p.numel() for p in model.parameters()))

39266


In [59]:
# necessary because zero padding would push towards learning all-zero delays
# we compute distance only between a chain of given length and the model's prediction up to that length;
# therefore useless predictions are made (computational waste), but we do not learn from them.

# def masked_mse_loss(predictions, targets, valid_lens):
#     seq_len = predictions.shape[1]
#     mask = torch.arange(seq_len, device=predictions.device).unsqueeze(0) < valid_lens.unsqueeze(1)  # [batch, seq_len]
#     mask = mask.unsqueeze(-1).expand_as(predictions)  # [batch, seq_len, 2]
#     diff2 = (predictions - targets) ** 2
#     return diff2[mask].mean()

# use a loss ("Huber loss"), less sensitive to outliers than MSE: defined as MSE if abs(x) < delta, MAE otherwise
# def masked_huber_loss(predictions, targets, valid_lens, delta=HUBER_LOSS_THRESH):
#     seq_len = predictions.shape[1]
#     mask = torch.arange(seq_len, device=predictions.device).unsqueeze(0) < valid_lens.unsqueeze(1)
#     mask = mask.unsqueeze(-1).expand_as(predictions)
#     loss_fn = nn.HuberLoss(delta=delta, reduction="none")
#     elementwise_loss = loss_fn(predictions, targets)
#     return elementwise_loss[mask].mean()

def masked_weighted_huber_loss(predictions, targets, valid_lens, delta=HUBER_LOSS_THRESH, pos_weight=4.5, delay_threshold_scaled=None):
    seq_len = predictions.shape[1]
    mask = torch.arange(seq_len, device=predictions.device).unsqueeze(0) < valid_lens.unsqueeze(1)
    mask = mask.unsqueeze(-1).expand_as(predictions)

    loss_fn = nn.HuberLoss(delta=delta, reduction="none")
    elementwise_loss = loss_fn(predictions, targets)

    weights = torch.ones_like(targets)
    weights[targets > delay_threshold_scaled.to(targets.device)] = pos_weight

    weighted_loss = elementwise_loss * weights
    return weighted_loss[mask].mean()

In [60]:
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGTH_DECAY)
# Adam optimizer with a sort of weigth decay incorporated (for regularization)

#### Training (manual early stopping)

In [61]:
best_val_loss = float("inf")
patience_counter = 0
early_stop = False

best_model_state = None

train_loss_history = []
val_loss_history = []
# Training Loop
for epoch in range(NUM_EPOCHS):
    print(f"Doing epoch {epoch}...")
    model.train()
    running_train_loss = 0.0

    for batch in train_loader: # load batches
        dense_feat, sparse_feat, labels, valid_lens, targets = [
            tensor.to(DEVICE) for tensor in batch
        ]

        # Zero the gradients from the previous step
        optimizer.zero_grad()

        # Forward pass
        #predictions = model(scale_tensors(dense_feat), sparse_feat)
        predictions = model(scale_tensors(dense_feat, mean=dense_mean, std=dense_std), sparse_feat)

        # loss
        #loss = criterion(predictions, scale_tensors(targets.float()))
        #loss = masked_mse_loss(predictions, scale_tensors(targets.float()), valid_lens)
        #loss = masked_huber_loss(predictions, scale_tensors(targets.float()), valid_lens)
        loss = masked_weighted_huber_loss(predictions, scale_tensors(targets.float()), valid_lens, delay_threshold_scaled=delay_threshold_scaled)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        running_train_loss += loss.item() * dense_feat.size(0)

    epoch_train_loss = running_train_loss / len(train_loader.dataset)

    # Validation Loop
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for batch in val_loader:
            dense_feat, sparse_feat, labels, valid_lens, targets = [
                tensor.to(DEVICE) for tensor in batch
            ]

            #predictions = model(dense_feat, sparse_feat)
            predictions = model(scale_tensors(dense_feat, mean=dense_mean, std=dense_std), sparse_feat)
            #loss = criterion(predictions, scale_tensors(targets.float()))
            #loss = masked_mse_loss(predictions, scale_tensors(targets.float()), valid_lens)
            #loss = masked_huber_loss(predictions, scale_tensors(targets.float()), valid_lens)
            loss = masked_weighted_huber_loss(predictions, scale_tensors(targets.float()), valid_lens, delay_threshold_scaled=delay_threshold_scaled)

            running_val_loss += loss.item() * dense_feat.size(0)

    epoch_val_loss = running_val_loss / len(val_loader.dataset)

    train_loss_history.append(epoch_train_loss)
    val_loss_history.append(epoch_val_loss)
    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}"
    )


    # Early stopping (if validattion loss doesn't improve for PATIENCE epochs)
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        patience_counter = 0
        best_model_state = copy.deepcopy(model.state_dict())
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch+1}")
            early_stop = True
            break

# After loop, load best (according to all validation history) model,
# unless there exists a previously saved one
if best_model_state is not None:
    model.load_state_dict(best_model_state)

print("Training and evaluation process completed successfully.")

Doing epoch 0...
Epoch [1/50] | Train Loss: 0.9205 | Val Loss: 1.5333
Doing epoch 1...
Epoch [2/50] | Train Loss: 0.9066 | Val Loss: 1.5346
Doing epoch 2...
Epoch [3/50] | Train Loss: 0.9015 | Val Loss: 1.5371
Doing epoch 3...
Epoch [4/50] | Train Loss: 0.8978 | Val Loss: 1.5332
Doing epoch 4...
Epoch [5/50] | Train Loss: 0.8951 | Val Loss: 1.5352
Doing epoch 5...
Epoch [6/50] | Train Loss: 0.8933 | Val Loss: 1.5366
Doing epoch 6...
Epoch [7/50] | Train Loss: 0.8918 | Val Loss: 1.5343
Doing epoch 7...
Epoch [8/50] | Train Loss: 0.8906 | Val Loss: 1.5372
Doing epoch 8...
Epoch [9/50] | Train Loss: 0.8895 | Val Loss: 1.5370
Early stopping triggered at epoch 9
Training and evaluation process completed successfully.


In [62]:
# took about an hour on Colab T4 GPU (free)

#### Testing / evaluation

In [63]:
model.eval()
all_preds, all_targets, all_labels, all_lens = [], [], [], []

with torch.no_grad():
    for batch in test_loader:
        dense_feat, sparse_feat, labels, valid_lens, targets = [
            t.to(DEVICE) for t in batch
        ]
        #preds = model(dense_feat, sparse_feat)
        preds = model(scale_tensors(dense_feat, mean=dense_mean, std=dense_std), sparse_feat)
        all_preds.append(preds.cpu())
        all_targets.append(targets.cpu())
        all_labels.append(labels.cpu()) # delay (> 15 min) labels, directly from dataset
        all_lens.append(valid_lens.cpu())  # flight chain lengths, directly from dataset

test_preds = torch.cat(all_preds, dim=0)
test_preds = unscale_tensors(test_preds)  # back to real minutes
test_targets = torch.cat(all_targets, dim=0)  # already unscaled ground truth, unchanged
test_labels = torch.cat(all_labels, dim=0)
test_lens = torch.cat(all_lens, dim=0)

test_metrics = compute_all_metrics(test_preds, test_targets, test_labels, test_lens, arr_threshold=15, dep_threshold=15)

print("\n=== Test Set Metrics ===")
for target in ["arr", "dep"]:
    print(f"\n{target.upper()} Regression:")
    for k, v in test_metrics[f"{target}_reg"].items():
        print(f"  {k}: {v:.4f}")
    print(f"{target.upper()} Classification:")
    for k, v in test_metrics[f"{target}_clf"].items():
        print(f"  {k}: {v:.4f}")


=== Test Set Metrics ===

ARR Regression:
  MSE: 2702.6165
  MAE: 27.2058
ARR Classification:
  Accuracy: 0.6563
  Precision: 0.3011
  Recall: 0.5928
  F1: 0.3993
  AUC: 0.6798

DEP Regression:
  MSE: 2483.1013
  MAE: 23.5636
DEP Classification:
  Accuracy: 0.6062
  Precision: 0.2894
  Recall: 0.6891
  F1: 0.4076
  AUC: 0.6889


In [64]:
# high accuracy (bc majority class) but low precision / recall / F1 -> the model is too conservative (predicts "on-time" too often)

In [65]:
# save metrics history and best model and dump them to a pickle file

epochs_run = (
    epoch + 1
)  # actual number of epochs completed (accounts for early stopping)

results = {
    "train_loss_history": train_loss_history,
    "val_loss_history": val_loss_history,
    "best_val_loss": best_val_loss,
    #'best_model_state': best_model_state,
    "test_metrics": test_metrics, # NB this
    "delay_mean": delay_mean,
    "delay_std": delay_std,
    "dense_mean": dense_mean,
    "dense_std": dense_std,
    # might be heavy, but if wanna test later:
    #'test_preds': test_preds,
    #'test_targets': test_targets,
    #'test_labels': test_labels,
    #'test_lens': test_lens,
    "epochs_run": epochs_run,
    "hyperparams": {
        "hidden_units": HIDDEN_UNITS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS,
        "fraction": fraction,
    },
}

with open("training_results.pkl", "wb") as f:
    pickle.dump(results, f)

# Save best model separately
torch.save(best_model_state, "best_model.pt")

In [72]:
#### Naive baselines

In [70]:
# Baseline check: what if we always predicted the mean delay for every sample?
mask_test = torch.arange(test_preds.shape[1]).unsqueeze(0) < test_lens.unsqueeze(1)  # [N, seq_len]

true_arr_valid = test_targets[:, :, 0][mask_test]
true_dep_valid = test_targets[:, :, 1][mask_test]

baseline_pred_arr = delay_mean[0].item()  # constant: train mean ARR delay
baseline_pred_dep = delay_mean[1].item()  # constant: train mean DEP delay

baseline_mae_arr = (true_arr_valid - baseline_pred_arr).abs().mean().item()
baseline_mae_dep = (true_dep_valid - baseline_pred_dep).abs().mean().item()

print(f"Baseline (predict train mean) MAE -- ARR: {baseline_mae_arr:.4f}, DEP: {baseline_mae_dep:.4f}")
print(f"Model MAE -- ARR: {test_metrics['arr_reg']['MAE']:.4f}, DEP: {test_metrics['dep_reg']['MAE']:.4f}")

Baseline (predict train mean) MAE -- ARR: 23.7377, DEP: 20.8284
Model MAE -- ARR: 27.2058, DEP: 23.5636


In [ ]:
# worse than baseline regression-wise when using *weighted* loss

In [71]:
# Naive baseline classifier: predicts "delayed" (1) for every sample
# using only the class prior (fraction of delayed samples in train set),
# via a fixed probability threshold at 0.5 -- equivalent to always predicting
# the majority class.

train_labels = train_dataset.tensors[2]  # [N, seq_len, 2]
train_valid_lens = train_dataset.tensors[3]  # [N]

seq_len = train_labels.shape[1]
mask_train = torch.arange(seq_len).unsqueeze(0) < train_valid_lens.unsqueeze(1)

label_arr_train = train_labels[:, :, 0][mask_train].float()
label_dep_train = train_labels[:, :, 1][mask_train].float()

# class prior = mean of binary labels (fraction delayed)
prior_arr = label_arr_train.mean().item()
prior_dep = label_dep_train.mean().item()

print(f"Prior P(delayed) -- ARR: {prior_arr:.4f}, DEP: {prior_dep:.4f}")

# naive prediction: always predict the majority class
naive_pred_arr = 1 if prior_arr > 0.5 else 0
naive_pred_dep = 1 if prior_dep > 0.5 else 0

mask_test = torch.arange(test_preds.shape[1]).unsqueeze(0) < test_lens.unsqueeze(1)
true_label_arr_test = test_labels[:, :, 0][mask_test].numpy()
true_label_dep_test = test_labels[:, :, 1][mask_test].numpy()

naive_preds_arr = np.full_like(true_label_arr_test, naive_pred_arr)
naive_preds_dep = np.full_like(true_label_dep_test, naive_pred_dep)

naive_acc_arr = accuracy_score(true_label_arr_test, naive_preds_arr)
naive_acc_dep = accuracy_score(true_label_dep_test, naive_preds_dep)

print(f"Naive baseline Accuracy -- ARR: {naive_acc_arr:.4f}, DEP: {naive_acc_dep:.4f}")

Prior P(delayed) -- ARR: 0.1966, DEP: 0.1987
Naive baseline Accuracy -- ARR: 0.8073, DEP: 0.8034


In [73]:
# Trivial baseline: always predict "delayed" (positive) for everyone
always_pos_preds_arr = np.ones_like(true_label_arr_test)
always_pos_preds_dep = np.ones_like(true_label_dep_test)

always_pos_acc_arr = accuracy_score(true_label_arr_test, always_pos_preds_arr)
always_pos_prec_arr = precision_score(true_label_arr_test, always_pos_preds_arr, zero_division=0)
always_pos_rec_arr = recall_score(true_label_arr_test, always_pos_preds_arr, zero_division=0)
always_pos_f1_arr = f1_score(true_label_arr_test, always_pos_preds_arr, zero_division=0)

always_pos_acc_dep = accuracy_score(true_label_dep_test, always_pos_preds_dep)
always_pos_prec_dep = precision_score(true_label_dep_test, always_pos_preds_dep, zero_division=0)
always_pos_rec_dep = recall_score(true_label_dep_test, always_pos_preds_dep, zero_division=0)
always_pos_f1_dep = f1_score(true_label_dep_test, always_pos_preds_dep, zero_division=0)

print("=== Trivial 'always predict delayed' baseline ===")
print(f"ARR -- Accuracy: {always_pos_acc_arr:.4f}, Precision: {always_pos_prec_arr:.4f}, Recall: {always_pos_rec_arr:.4f}, F1: {always_pos_f1_arr:.4f}")
print(f"DEP -- Accuracy: {always_pos_acc_dep:.4f}, Precision: {always_pos_prec_dep:.4f}, Recall: {always_pos_rec_dep:.4f}, F1: {always_pos_f1_dep:.4f}")

=== Trivial 'always predict delayed' baseline ===
ARR -- Accuracy: 0.1927, Precision: 0.1927, Recall: 1.0000, F1: 0.3232
DEP -- Accuracy: 0.1966, Precision: 0.1966, Recall: 1.0000, F1: 0.3286


In [66]:
test_metrics_var_thresh = compute_all_metrics(test_preds, test_targets, test_labels, test_lens)

Using best threshold for arrival metrics...
Using best threshold for departure metrics...


In [67]:
print("\n=== Test Set Metrics (variable threshoold)===")
for target in ["arr", "dep"]:
    print(f"\n{target.upper()} Regression:")
    for k, v in test_metrics_var_thresh[f"{target}_reg"].items():
        print(f"  {k}: {v:.4f}")
    print(f"{target.upper()} Classification:")
    for k, v in test_metrics_var_thresh[f"{target}_clf"].items():
        print(f"  {k}: {v:.4f}")


=== Test Set Metrics (variable threshoold)===

ARR Regression:
  MSE: 2702.6165
  MAE: 27.2058
ARR Classification:
  Accuracy: 0.6349
  Precision: 0.2922
  Recall: 0.6285
  F1: 0.3989
  AUC: 0.6798

DEP Regression:
  MSE: 2483.1013
  MAE: 23.5636
DEP Classification:
  Accuracy: 0.6436
  Precision: 0.3045
  Recall: 0.6328
  F1: 0.4111
  AUC: 0.6889


In [68]:
results_var_thresh = {
    "train_loss_history": train_loss_history,
    "val_loss_history": val_loss_history,
    "best_val_loss": best_val_loss,
    #'best_model_state': best_model_state,
    "test_metrics": test_metrics_var_thresh,
    "delay_mean": delay_mean,
    "delay_std": delay_std,
    "dense_mean": dense_mean,
    "dense_std": dense_std,
    # might be heavy, but if wanna test later:
    #'test_preds': test_preds,
    #'test_targets': test_targets,
    #'test_labels': test_labels,
    #'test_lens': test_lens,
    "epochs_run": epochs_run,
    "hyperparams": {
        "hidden_units": HIDDEN_UNITS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS,
        "fraction": fraction,
    },
}

with open("training_results_var_thresh.pkl", "wb") as f:
    pickle.dump(results_var_thresh, f)

In [69]:
# recall > precision so false negatives < false positives (not-so-conservative...)